## Compress

Observability systems may receive a very large amount of data every day. In addition to data governance, what other methods can help reduce disk usage? One recommended approach is to use the ZSTD compression algorithm to compress data efficiently and reduce storage consumption.

The logs have already been imported into the `otel_logs` table. That table uses the default compression algorithm, LZ4F. To compare it with the default `otel_logs` table, create a new table named `otel_logs_zstd`:

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [2]:
lab.execute(r"""
CREATE TABLE `otel_logs_zstd` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200) NULL,
  `trace_id` varchar(200) NULL,
  `span_id` text NULL,
  `severity_number` int NULL,
  `severity_text` text NULL,
  `body` text NULL,
  `resource_attributes` variant NULL,
  `log_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_trace_id (`trace_id`) USING INVERTED,
  INDEX idx_span_id (`span_id`) USING INVERTED,
  INDEX idx_severity_number (`severity_number`) USING INVERTED,
  INDEX idx_body (`body`) USING INVERTED PROPERTIES("lower_case" = "true", "parser" = "unicode", "support_phrase" = "true"),
  INDEX idx_severity_text (`severity_text`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_log_attributes (`log_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)
(PARTITION p20260908 VALUES [('2026-09-08 00:00:00'), ('2026-09-09 00:00:00')),
PARTITION p20260909 VALUES [('2026-09-09 00:00:00'), ('2026-09-10 00:00:00')),
PARTITION p20260910 VALUES [('2026-09-10 00:00:00'), ('2026-09-11 00:00:00')),
PARTITION p20260911 VALUES [('2026-09-11 00:00:00'), ('2026-09-12 00:00:00')),
PARTITION p20260912 VALUES [('2026-09-12 00:00:00'), ('2026-09-13 00:00:00')),
PARTITION p20260913 VALUES [('2026-09-13 00:00:00'), ('2026-09-14 00:00:00')),
PARTITION p20260914 VALUES [('2026-09-14 00:00:00'), ('2026-09-15 00:00:00')),
PARTITION p20260915 VALUES [('2026-09-15 00:00:00'), ('2026-09-16 00:00:00')),
PARTITION p20260916 VALUES [('2026-09-16 00:00:00'), ('2026-09-17 00:00:00')),
PARTITION p20260917 VALUES [('2026-09-17 00:00:00'), ('2026-09-18 00:00:00')),
PARTITION p20260918 VALUES [('2026-09-18 00:00:00'), ('2026-09-19 00:00:00')),
PARTITION p20260919 VALUES [('2026-09-19 00:00:00'), ('2026-09-20 00:00:00')))
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-10",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "10",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"enable_single_replica_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728",
"compression" = "zstd"
);
""", title='ZSTD-compressed log table')


0

Note the final property, `"compression" = "zstd"`. It specifies that the ZSTD compression algorithm is used.

Next, insert the data from `otel_logs` into `otel_logs_zstd`:

In [3]:
lab.execute("INSERT INTO otel_logs_zstd SELECT * FROM otel_logs")


97055

Because the data files are not large, this example is intended only to show the difference. In scenarios with larger data volumes, ZSTD can provide better compression results:

<img src="images/compress.png" alt="Comparison of LZ4F and ZSTD compression in Doris" style="max-width:100%;height:auto;">

## Materialized Views

In metrics scenarios, if you need to analyze the maximum value of certain `metric_name` values over a historical period, you can query the `otel_metrics_gauge` table directly. However, this table stores the original metrics data, and direct aggregation can be inefficient. Doris materialized views can be used to accelerate these queries.

The `otel_metrics_gauge` table has already been created. You only need to create a materialized view on this table to accelerate the required query:

In [4]:
lab.execute(r"""
CREATE MATERIALIZED VIEW mv_for_metrics AS
SELECT metric_name AS mn, MAX(value) AS max_value
FROM otel_metrics_gauge
GROUP BY metric_name;
""", title='Materialized view for metric maxima')


0

After the materialized view is created, you can query the original table without needing to remember that the materialized view exists:

In [5]:
lab.sql(r"""
SELECT metric_name, MAX(value) FROM otel_metrics_gauge GROUP BY metric_name LIMIT 5;
""", title='Metric maximum query')


metric_name,MAX(value)
scrape_samples_post_metric_relabeling,1455.0
up,1.0
doris_be_thread_pool_max_queue_size,2147483647.0
doris_be_memory_pswpin,90120.0
doris_be_memory_pgpgout,13402708.0
doris_be_hdfs_file_open_reading,0.0
doris_be_tablet_version_num_distribution_median,1.0
doris_be_memory_allocated_bytes,1824768000.0
doris_be_broker_count,0.0
doris_be_thread_pool_max_threads,2048.0


,metric_name,MAX(value)
0,scrape_samples_post_metric_relabeling,1.455000e+03
1,up,1.000000e+00
2,doris_be_thread_pool_max_queue_size,2.147484e+09
3,doris_be_memory_pswpin,9.012000e+04
4,doris_be_memory_pgpgout,1.340271e+07
...,...,...
178,doris_fe_tablet_max_compaction_score,1.500000e+01
179,doris_fe_plan_num,1.000000e+00
180,doris_fe_report_queue_size,0.000000e+00
181,doris_fe_statistics_empty_table_num,5.000000e+00


This statement will hit the materialized view. Because the materialized view already contains aggregated data, the query will be accelerated. You can use `EXPLAIN` to confirm this:

<img src="images/explain.png" alt="EXPLAIN output showing that the materialized view is used" style="max-width:100%;height:auto;">

## Inverted Indexes

You may also have noticed that many of the preceding table definitions contain statements such as the following. These statements create inverted indexes:

The original log table contains the following inverted indexes:

```sql
...
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_trace_id (`trace_id`) USING INVERTED,
  ...
```


How can you determine whether these indexes are being used and whether they reduce the amount of data that must be scanned?

Consider a simple scenario where you need to retrieve detailed logs whose `service_name` equals a specific value. The following statement can be generated quickly. The first two session variables enable profiling and set the profile level:

In [6]:
lab.execute("SET enable_profile = true")
lab.sql(r"""
SELECT * FROM otel_logs WHERE service_name = 'load-generator' LIMIT 10
""", title="Query with profile collection enabled")


timestamp,service_name,service_instance_id,trace_id,span_id,severity_number,severity_text,body,resource_attributes,log_attributes,scope_name,scope_version


,timestamp,service_name,service_instance_id,trace_id,span_id,severity_number,severity_text,body,resource_attributes,log_attributes,scope_name,scope_version


Use the following statement to find the profile ID for the preceding query:

In [7]:
lab.sql(r"""
SHOW QUERY PROFILE
""", title='Recent query profiles')


Profile ID,Task Type,Start Time,End Time,Total,Task State,User,Default Catalog,Default Db,Sql Statement
88e5c13d303b4385-8b6814758777d498,QUERY,2026-09-24 10:32:24,2026-09-24 10:32:25,997ms,OK,root,internal,otel,SELECT * FROM otel_logs WHERE service_name = 'load-generator' LIMIT 10


,Profile ID,Task Type,Start Time,End Time,Total,Task State,User,Default Catalog,Default Db,Sql Statement
0,88e5c13d303b4385-8b6814758777d498,QUERY,2026-09-24 10:32:24,2026-09-24 10:32:25,997ms,OK,root,internal,otel,SELECT * FROM otel_logs WHERE service_name = '...


On the FE that executed the query, run the following command:

In [9]:
PROFILE_ID = "88e5c13d303b4385-8b6814758777d498	"
assert not PROFILE_ID.startswith("<"), "Replace PROFILE_ID with the ID shown by SHOW QUERY PROFILE."

lab.shell(f"""set -euo pipefail
curl -uroot: "http://127.0.0.1:8030/api/profile/text?query_id=88e5c13d303b4385-8b6814758777d498"
""", title="Fetch the Doris query profile")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 25 16158   25  4136    0     0  17719      0 --:--:-- --:--:-- --:--:-- 17675
100 16158  100 16158    0     0  69171      0 --:--:-- --:--:-- --:--:-- 69051
Summary:
   - Profile ID: 88e5c13d303b4385-8b6814758777d498
   - Task Type: QUERY
   - Start Time: 2026-09-24 10:32:24
   - End Time: 2026-09-24 10:32:25
   - Total: 997ms
   - Task State: OK
   - User: root
   - Default Catalog: internal
   - Default Db: otel
   - Sql Statement: SELECT * FROM otel_logs WHERE service_name = 'load-generator' LIMIT 10
   - Distributed Plan: N/A
Execution Summary:
   - Workload Group: normal
   - Parse SQL Time: 3ms
   - Plan Time: 136ms
     - Garbage Collect During Plan Time: 0ms
     - Nereids Lock Table Time: 84ms
     - Nereids Analysis Time: 2ms
     - Nereids 

CompletedProcess(args=['/bin/bash', '-lc', 'set -euo pipefail\ncurl -uroot: "http://127.0.0.1:8030/api/profile/text?query_id=88e5c13d303b4385-8b6814758777d498"'], returncode=0, stdout='  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\n  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\n 25 16158   25  4136    0     0  17719      0 --:--:-- --:--:-- --:--:-- 17675\n100 16158  100 16158    0     0  69171      0 --:--:-- --:--:-- --:--:-- 69051\nSummary:\n   - Profile ID: 88e5c13d303b4385-8b6814758777d498\n   - Task Type: QUERY\n   - Start Time: 2026-09-24 10:32:24\n   - End Time: 2026-09-24 10:32:25\n   - Total: 997ms\n   - Task State: OK\n   - User: root\n   - Default Catalog: internal\n   - Default Db: otel\n   - Sql Statement: SELECT * FROM otel_logs WHERE service_name = \'load-generator\' LIMIT 10\n   - Distributed Plan: N/A\nExecution Summary:\n

Search for the keyword `RowsInvertedIndexFiltered`. Information similar to the following is displayed:

```text
RowsInvertedIndexFiltered: 11.702K (11702)
```

This indicates that 11,702 rows were filtered out by the inverted index. The index filtered part of the data and improved query efficiency.